# 86 — SmolVLA P&P `(1,2,3)` uncertainty analysis

Analyze worker 85 on its exact 400 matched identities. This notebook reports paired SR change versus the historical stock SmolVLA arm and asks whether U10/U20/U50 predicts failure of the `(1,2,3)` refinement rollout. It evaluates each Euler step separately, an equal-step mean, and a `(3,1,1)` disagreement-pair-weighted mean.

In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## Load exact-matched rollout rows

In [ ]:
from pathlib import Path
import io
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm
from sklearn.metrics import roc_curve

from analysis.horizon_diagnostics import failure_auc_table, _download_with_retry
from analysis.smolvla_libero import pair_stock_refinement
from analysis.suffix_sensitivity import summarize_pair
from pnp.diversity import DIVERSITY_PAIR_KEYS
from pnp.experiments import SMOLVLA_LIBERO_EXPERIMENT, build_smolvla_libero_methods
from pnp.smolvla_followup_experiments import (
    SMOLVLA_SCHEDULE_EXPERIMENT, SMOLVLA_SCHEDULE_K_BY_STEP,
    SMOLVLA_SCHEDULE_STEPS, build_smolvla_schedule_method)
from pnp.store import SupabaseStore

EXPECTED_IDENTITIES = 400
HORIZONS = (10, 20, 50)
N_BOOT = 3000
OUTPUT = Path('smolvla_pnp_steps123_k311_analysis')
CACHE = OUTPUT / 'cache'
OUTPUT.mkdir(exist_ok=True); CACHE.mkdir(exist_ok=True)
store = SupabaseStore()

def completed_rows(experiment, method, config):
    config_hash = store.config_hash(store._logical_key(method, config))
    rows = pd.DataFrame(store.fetch_all(
        'rollouts', '*', configure=lambda query: query.eq('experiment', experiment)
        .eq('method', method).eq('config_hash', config_hash).eq('status', 'completed'),
        order_by=('rollout_id',)))
    if rows.duplicated(DIVERSITY_PAIR_KEYS).any():
        raise ValueError(f'duplicate identities for {experiment}/{method}')
    return rows, config_hash

schedule_method, schedule_config = build_smolvla_schedule_method()
schedule_rows, schedule_hash = completed_rows(
    SMOLVLA_SCHEDULE_EXPERIMENT, schedule_method, schedule_config)
stock_method, stock_config = build_smolvla_libero_methods()[0]
stock_rows, stock_hash = completed_rows(
    SMOLVLA_LIBERO_EXPERIMENT, stock_method, stock_config)
wanted = schedule_rows[DIVERSITY_PAIR_KEYS]
stock_rows = stock_rows.merge(wanted, on=DIVERSITY_PAIR_KEYS, validate='one_to_one')
if len(schedule_rows) != EXPECTED_IDENTITIES or len(stock_rows) != EXPECTED_IDENTITIES:
    raise ValueError(
        f'expected {EXPECTED_IDENTITIES} exact rows; schedule={len(schedule_rows)}, stock={len(stock_rows)}')
if schedule_rows.ahats_path.isna().any():
    raise ValueError('worker 85 rows are missing uncertainty artifacts')
paired = pair_stock_refinement(stock_rows, schedule_rows)
print({'schedule_experiment': SMOLVLA_SCHEDULE_EXPERIMENT,
       'exact_matched_identities': len(paired), 'steps': SMOLVLA_SCHEDULE_STEPS,
       'pairs_per_step': SMOLVLA_SCHEDULE_K_BY_STEP,
       'schedule_config_hash': schedule_hash, 'stock_config_hash': stock_hash})

## Decode step-specific uncertainty

In [ ]:
KEY = re.compile(r'^c(?P<chunk>\d+)_s(?P<step>\d+)_u_time$')
cache_file = CACHE / f'step_records_{schedule_hash[:12]}.pkl'
if cache_file.exists():
    records = pd.read_pickle(cache_file)
    print('Loaded decoded worker-85 artifacts from cache.')
else:
    decoded = []
    for row in tqdm(schedule_rows.to_dict('records'), desc='worker-85 artifacts'):
        payload = _download_with_retry(store, str(row['ahats_path']))
        with np.load(io.BytesIO(payload)) as archive:
            for key in archive.files:
                match = KEY.match(key)
                if not match:
                    continue
                u_time = np.asarray(archive[key], dtype=float).reshape(-1)
                iter_key = key.removesuffix('_u_time') + '_u_iter_time'
                iter_time = np.asarray(archive[iter_key], dtype=float)
                if len(u_time) != 50 or iter_time.ndim != 2 or iter_time.shape[1] != 50:
                    raise ValueError(f'{row["rollout_id"]}/{key}: malformed uncertainty profile')
                step = int(match.group('step'))
                expected_pairs = dict(zip(SMOLVLA_SCHEDULE_STEPS, SMOLVLA_SCHEDULE_K_BY_STEP))[step]
                if iter_time.shape[0] != expected_pairs:
                    raise ValueError(
                        f'{row["rollout_id"]}/{key}: expected {expected_pairs} pairs, found {iter_time.shape[0]}')
                item = {'rollout_id': row['rollout_id'], 'suite': row['suite'],
                        'success': bool(row['success']),
                        'chunk_idx': int(match.group('chunk')), 'euler_step': step}
                for horizon in HORIZONS:
                    item[f'u{horizon}'] = float(u_time[:horizon].mean())
                decoded.append(item)
    records = pd.DataFrame(decoded)
    records.to_pickle(cache_file)

observed_steps = tuple(sorted(records.euler_step.unique()))
if observed_steps != tuple(SMOLVLA_SCHEDULE_STEPS):
    raise ValueError(f'expected steps {SMOLVLA_SCHEDULE_STEPS}, found {observed_steps}')
counts = records.groupby(['rollout_id', 'chunk_idx']).euler_step.nunique()
if not counts.eq(len(SMOLVLA_SCHEDULE_STEPS)).all():
    raise ValueError('at least one chunk is missing a selected Euler-step profile')
print({'episodes': records.rollout_id.nunique(), 'chunks': len(counts),
       'step_records': len(records), 'steps': observed_steps})

## Matched success rate and per-suite delta

In [ ]:
overall_sr, suite_sr = summarize_pair(paired)
display(overall_sr); display(suite_sr)
overall_sr.to_csv(OUTPUT / 'overall_paired_sr.csv', index=False)
suite_sr.to_csv(OUTPUT / 'suite_paired_sr.csv', index=False)
labels = suite_sr.suite.str.removeprefix('libero_')
x = np.arange(len(suite_sr)); width = .36
fig, axes = plt.subplots(2, 1, figsize=(13, 10), constrained_layout=True)
axes[0].bar(x-width/2, suite_sr.baseline_sr_pct, width, label='historical stock')
axes[0].bar(x+width/2, suite_sr.condition_sr_pct, width, label='P&P (1,2,3), K=(3,1,1)')
axes[0].set_xticks(x, labels, rotation=25, ha='right')
axes[0].set(ylabel='Success rate (%)', ylim=(0,105), title='Exact-matched success rate')
axes[0].legend(); axes[0].grid(axis='y', alpha=.2)
delta = suite_sr.condition_minus_baseline_pp.to_numpy(float)
axes[1].bar(x, delta, color=np.where(delta >= 0, '#54A24B', '#E45756'))
axes[1].errorbar(x, delta, yerr=np.vstack((
    delta-suite_sr.delta_ci_low_pp, suite_sr.delta_ci_high_pp-delta)),
    fmt='none', ecolor='black', capsize=3)
axes[1].axhline(0, color='black'); axes[1].set_xticks(x, labels, rotation=25, ha='right')
axes[1].set(ylabel='P&P minus stock SR (percentage points)', title='Paired SR change by suite')
axes[1].grid(axis='y', alpha=.2)
fig.savefig(OUTPUT / 'matched_sr_and_delta.png', dpi=180); plt.show()

## Step-specific, equal-step, and pair-weighted uncertainty

`weighted` uses weights `(3,1,1)`, so it is the mean over all five logged disagreement pairs. `equal` gives each Euler step equal influence. AUC labels failure of the worker-85 refinement rollout, since its trajectory can diverge from stock.

In [ ]:
def make_features(source, suffix):
    per_step = source.groupby(['rollout_id', 'euler_step'])[[f'u{h}' for h in HORIZONS]].mean()
    wide = per_step.unstack('euler_step')
    wide.columns = [f'{metric}_step{step}_{suffix}' for metric, step in wide.columns]
    wide = wide.reset_index()
    metadata = schedule_rows[DIVERSITY_PAIR_KEYS + ['rollout_id', 'success']].copy()
    result = metadata.merge(wide, on='rollout_id', validate='one_to_one')
    weights = dict(zip(SMOLVLA_SCHEDULE_STEPS, SMOLVLA_SCHEDULE_K_BY_STEP))
    for horizon in HORIZONS:
        columns = [f'u{horizon}_step{step}_{suffix}' for step in SMOLVLA_SCHEDULE_STEPS]
        result[f'u{horizon}_equal_{suffix}'] = result[columns].mean(axis=1)
        result[f'u{horizon}_weighted_{suffix}'] = sum(
            weights[step] * result[f'u{horizon}_step{step}_{suffix}']
            for step in SMOLVLA_SCHEDULE_STEPS) / sum(weights.values())
    return result

episode_features = make_features(records, 'episode')
first_features = make_features(records[records.chunk_idx.eq(0)], 'first_chunk')
features = episode_features.merge(
    first_features[['rollout_id'] + [c for c in first_features if c.startswith('u')]],
    on='rollout_id', validate='one_to_one')
score_columns = [
    f'u{h}_{summary}_{timing}'
    for timing in ('episode', 'first_chunk') for h in HORIZONS
    for summary in ('step1', 'step2', 'step3', 'equal', 'weighted')]
auc_table = failure_auc_table(features, score_columns, n_boot=N_BOOT)
auc_table.to_csv(OUTPUT / 'step_specific_failure_auc.csv', index=False)
print('Pooled failure AUC')
display(auc_table[auc_table.suite.eq('pooled')])
print('Per-suite failure AUC (all scores are retained in the CSV)')
display(auc_table[~auc_table.suite.eq('pooled')])

In [ ]:
summary_order = ('step1', 'step2', 'step3', 'equal', 'weighted')
summary_labels = ('step 1 (K=3)', 'step 2 (K=1)', 'step 3 (K=1)',
                  'equal-step mean', '(3,1,1)-weighted mean')
pooled = auc_table[auc_table.suite.eq('pooled')].set_index('score_name')
fig, axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True, sharey=True)
for axis, horizon in zip(axes, HORIZONS):
    names = [f'u{horizon}_{name}_episode' for name in summary_order]
    group = pooled.reindex(names)
    center = group.failure_auc.to_numpy(float)
    y = np.arange(len(names))
    axis.errorbar(center, y, xerr=np.vstack((center-group.auc_ci_low, group.auc_ci_high-center)),
                  fmt='o', capsize=3)
    axis.axvline(.5, color='black', linestyle='--')
    axis.set_yticks(y, summary_labels); axis.set_xlim(0, 1)
    axis.set(xlabel='Failure ROC-AUC (95% bootstrap CI)', title=f'U{horizon}, full episode')
    axis.grid(axis='x', alpha=.2)
fig.savefig(OUTPUT / 'pooled_step_and_aggregate_auc.png', dpi=180); plt.show()

weighted_scores = [f'u{h}_weighted_episode' for h in HORIZONS]
plot_auc = auc_table[~auc_table.suite.eq('pooled') & auc_table.score_name.isin(weighted_scores)]
suites = sorted(plot_auc.suite.unique())
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
for offset, horizon in zip((-.18, 0, .18), HORIZONS):
    score = f'u{horizon}_weighted_episode'
    group = plot_auc[plot_auc.score_name.eq(score)].set_index('suite').reindex(suites)
    valid = group.failure_auc.notna().to_numpy(); center = group.failure_auc.to_numpy()[valid]
    axes[0].errorbar(center, np.arange(len(suites))[valid]+offset,
        xerr=np.vstack((center-group.auc_ci_low.to_numpy()[valid],
                        group.auc_ci_high.to_numpy()[valid]-center)),
        fmt='o', capsize=3, label=f'weighted U{horizon}')
axes[0].set_yticks(np.arange(len(suites)), [s.removeprefix('libero_') for s in suites])
axes[0].axvline(.5, color='black', linestyle='--')
axes[0].set(xlim=(0,1), xlabel='Failure ROC-AUC (95% bootstrap CI)',
            title='Pair-weighted uncertainty AUC by suite')
axes[0].legend(); axes[0].grid(axis='x', alpha=.2)
failure = ~features.success.astype(bool).to_numpy()
for horizon, score in zip(HORIZONS, weighted_scores):
    fpr, tpr, _ = roc_curve(failure, features[score])
    axes[1].plot(fpr, tpr, label=f'U{horizon}: {pooled.loc[score, "failure_auc"]:.3f}')
axes[1].plot([0,1], [0,1], 'k--', label='chance')
axes[1].set(xlabel='False-positive rate', ylabel='True-positive rate',
            title='Pooled pair-weighted failure ROC')
axes[1].legend(); axes[1].grid(alpha=.2)
fig.savefig(OUTPUT / 'weighted_auc_by_suite_and_roc.png', dpi=180); plt.show()

## Compact comparison

In [ ]:
comparison = pooled.reset_index()
comparison = comparison[comparison.score_name.str.endswith('_episode')][
    ['score_name', 'episodes', 'failures', 'failure_auc', 'auc_ci_low', 'auc_ci_high']]
comparison['horizon'] = comparison.score_name.str.extract(
    r'^u(10|20|50)', expand=False).astype(int)
comparison['summary'] = comparison.score_name.str.extract(
    r'_((?:step[123])|equal|weighted)_episode$', expand=False)
comparison = comparison.sort_values(['horizon', 'summary'])
display(comparison)
comparison.to_csv(OUTPUT / 'pooled_auc_comparison.csv', index=False)
print({'stock_sr_pct': 100*paired.baseline_success.mean(),
       'schedule_sr_pct': 100*paired.condition_success.mean(),
       'schedule_minus_stock_pp': 100*(paired.condition_success.mean()-paired.baseline_success.mean()),
       'outputs': str(OUTPUT.resolve())})